# Planning a robot-only service visit: mission M1 at turbine T07

**Part of the *Robotics for Offshore Wind* Virtual Research Building (AICOR, University of Bremen), prepared for the Siemens Energy Innovation Lab, Cuxhaven.**

Mission M1 is a service visit at one turbine of the Windanker wind farm with **no people in the turbine**:
- a service vessel lands its gangway;
- drones survey the blades, test the lightning receptors and fly in a filter kit;
- a legged robot with an arm (ANYmal D) and an inspection robot (Spot) are hoisted into the nacelle;
- the nacelle's robotic crane brings the parts.

The plan has 18 steps with actors, shared resources (hoist, hatch, gangway, airspaces) and dependencies. This
notebook schedules it, shows which steps set the length of the visit, checks it against weather limits, and lets
you change the plan.

**Read the results with these limits in mind.**
- **M1 is a task plan, not a simulation.** No step is executed here.
- **Durations and weather limits are illustrative** (from the twin), except where the twin marks a vendor figure.
- The twin is **reconstructed from public and second-hand sources**. It contains no Siemens Energy service data.

In [ ]:
import os, sys, copy, subprocess
TWIN = os.path.join(os.environ.get("REPO_DIR", ".."), "CuxhavenSiteTwin")
if not os.path.isdir(TWIN):
    TWIN = os.path.abspath(os.path.join("..", "CuxhavenSiteTwin"))
sys.path.insert(0, os.path.join(TWIN, "offshore", "missions"))
import m1_service_visit as M            # the twin's plan (offshore/missions/m1_service_visit.py)
commit = subprocess.run(["git", "-C", TWIN, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("twin commit", commit, "|", len(M.STEPS), "steps |", len(M.ACTORS), "actors")
for k, v in M.ACTORS.items():
    print(f"  {k:6s} {v}")

## 1. The scheduler, for any version of the plan

The twin's scheduler gives every step the earliest start at which its predecessors are done and none of its actors
or resources is busy. The function below does the same for an edited plan. The cell checks that it gives the twin's
schedule for the original plan.

In [ ]:
def schedule(steps):
    start, end, busy, done = {}, {}, {}, set()
    while len(done) < len(steps):
        progressed = False
        for sid, (title, actors, dur, after, res, note) in steps.items():
            if sid in done or any(a not in done for a in after):
                continue
            t = max([end[a] for a in after], default=0)
            moved = True
            while moved:
                moved = False
                for k in res + actors:
                    for (a, b) in busy.get(k, []):
                        if t < b and t + dur > a:
                            t, moved = b, True
            start[sid], end[sid] = t, t + dur
            for k in res + actors:
                busy.setdefault(k, []).append((t, t + dur))
            done.add(sid); progressed = True
        if not progressed:
            raise ValueError("the plan has a dependency on a missing or circular step")
    return start, end

BASE = copy.deepcopy(M.STEPS)
s0, e0 = schedule(BASE)
print("same schedule as the twin:", (s0, e0) == M.schedule(), "| total", max(e0.values()), "min")

In [ ]:
import matplotlib.pyplot as plt

def gantt(steps, start, end, title):
    total = max(end.values())
    crit = critical(steps)
    fig, ax = plt.subplots(figsize=(10, 0.32 * len(steps) + 1.2))
    for i, sid in enumerate(steps):
        c = "#C8102E" if sid in crit else "#0B3D6B"
        ax.barh(i, end[sid] - start[sid], left=start[sid], color=c, alpha=.85)
        ax.text(end[sid] + 2, i, f"{sid} {steps[sid][0][:58]} ({', '.join(steps[sid][1])})", va="center", fontsize=7)
    ax.set_yticks([]); ax.invert_yaxis(); ax.set_xlim(0, total * 1.75); ax.set_xlabel("minutes from arrival at the turbine")
    ax.set_title(f"{title}: {total} min; red = steps that set the length of the visit", fontsize=10, loc="left")
    plt.tight_layout(); plt.show()

def critical(steps, delta=1):
    # a step is critical (no slack) if making it 1 min longer makes the whole visit longer
    base = max(schedule(steps)[1].values())
    out = set()
    for sid in steps:
        st = copy.deepcopy(steps); t = list(st[sid]); t[2] += delta; st[sid] = tuple(t)
        if max(schedule(st)[1].values()) > base:
            out.add(sid)
    return out

gantt(BASE, s0, e0, "M1 as planned in the twin")
crit = critical(BASE)
print("no slack (a delay here delays the visit):", ", ".join(f"{k} {BASE[k][0][:40]}" for k in BASE if k in crit))
print("\nwith slack:", ", ".join(f"{k} {BASE[k][0][:40]}" for k in BASE if k not in crit))

**What to look for.** Only the red steps set the length of the visit; speeding up any other step changes nothing. Check the two lists printed above the chart rather than guessing. In the twin's plan most steps of the legged robot and the nacelle crane have no slack, because they share one hoist, one hatch and one robot. The inspection drones and the incoming kit delivery have slack. A resident robot that skips the hoist-in step changes that coupling: try it below.

## 2. Weather: can the visit happen today?

The twin records four limits (`M.LIMITS`, illustrative). We attach them to steps by the resource the step uses. The
mapping is ours, not the twin's:

| Limit | Applies to steps that use |
|---|---|
| significant wave height for the gangway | `gangway` |
| significant wave height for robot transfer | steps where the robots cross the gangway |
| wind for the cargo drone | `airspace B` (cargo drone) |
| wind for the nacelle hoist | `hoist` |

The inspection and contact drones (`airspace A`) have no limit in the twin, so they are shown as unchecked.

In [ ]:
print(M.LIMITS)

def weather_check(steps, hs_m, wind_ms):
    L = M.LIMITS
    rows = []
    for sid, (title, actors, dur, after, res, note) in steps.items():
        why = []
        if "gangway" in res and hs_m > L["gangway_Hs_m"]:
            why.append(f"Hs {hs_m} m > gangway {L['gangway_Hs_m']} m")
        if "gangway" in res and set(actors) & {"A1", "S1"} and hs_m > L["robot_transfer_Hs_m"]:
            why.append(f"Hs {hs_m} m > robot transfer {L['robot_transfer_Hs_m']} m")
        if "airspace B" in res and wind_ms > L["cargo_drone_wind_ms"]:
            why.append(f"wind {wind_ms} m/s > cargo drone {L['cargo_drone_wind_ms']} m/s")
        if "hoist" in res and wind_ms > L["nacelle_hoist_wind_ms"]:
            why.append(f"wind {wind_ms} m/s > hoist {L['nacelle_hoist_wind_ms']} m/s")
        unchecked = "airspace A" in res
        rows.append((sid, title, "; ".join(why), unchecked))
    blocked = [r for r in rows if r[2]]
    print(f"Hs {hs_m} m, wind {wind_ms} m/s: {'visit can go ahead' if not blocked else f'{len(blocked)} step(s) blocked'}")
    for sid, title, why, unchecked in rows:
        if why or unchecked:
            print(f"  {sid} {title[:60]:60s} {why or 'no limit in the twin (unchecked)'}")

weather_check(BASE, hs_m=1.5, wind_ms=10)
weather_check(BASE, hs_m=1.5, wind_ms=13)

## 3. Change the plan

These edits are what-ifs made in this notebook. They are not part of the twin's plan:
- **No pump-motor exchange:** steps S13 and S14 are dropped, for when the inspection does not confirm the bearing noise.
- **Resident robot:** the legged robot already waits in the nacelle from the last visit, so it is not hoisted in (S07 dropped).
- **Robot work speed:** scales the steps in which the legged robot works. 1.5 means 50 % slower than planned.

Press **Run**. Without widgets, edit the last cell.

In [ ]:
def edited(motor=True, resident=False, robot_speed=1.0):
    st = copy.deepcopy(BASE)
    def drop(sid, replace_with):
        st.pop(sid)
        for k, (t, a, d, after, r, n) in list(st.items()):
            new = []
            for x in after:
                for y in ([replace_with] if isinstance(replace_with, str) else replace_with) if x == sid else [x]:
                    if y and y not in new:
                        new.append(y)
            st[k] = (t, a, d, new, r, n)
    if not motor:
        drop("S14", ["S12"]); drop("S13", [])
    if resident:
        t, a, d, after, r, n = st["S05"]; st["S05"] = (t.replace("A1 and S1", "S1"), ["S1"], d, after, r, n)
        drop("S07", ["S05", "S06"])
    for k, (t, a, d, after, r, n) in list(st.items()):
        if "A1" in a and robot_speed != 1.0:
            st[k] = (t, a, round(d * robot_speed), after, r, n)
    return st

def run(motor=True, resident=False, robot_speed=1.0, hs_m=1.5, wind_ms=10.0):
    st = edited(motor, resident, robot_speed)
    s, e = schedule(st)
    print(f"total {max(e.values())} min (as planned: {max(e0.values())} min) | steps {len(st)}")
    gantt(st, s, e, "Your plan")
    weather_check(st, hs_m, wind_ms)

try:
    import ipywidgets as W
    from IPython.display import display
    w = dict(motor=W.Checkbox(value=True, description="Pump-motor exchange"),
             resident=W.Checkbox(value=False, description="Resident robot"),
             robot_speed=W.FloatSlider(value=1.0, min=0.5, max=2.0, step=0.1, description="Robot time ×"),
             hs_m=W.FloatSlider(value=1.5, min=0.5, max=4.0, step=0.25, description="Hs (m)"),
             wind_ms=W.FloatSlider(value=10.0, min=0.0, max=20.0, step=1.0, description="Wind (m/s)"))
    out = W.Output(); btn = W.Button(description="Run", button_style="primary")
    def go(_):
        with out:
            out.clear_output(); run(**{k: v.value for k, v in w.items()})
    btn.on_click(go)
    display(W.VBox(list(w.values()) + [btn, out]))
except ImportError:
    print("ipywidgets is not installed; use the next cell")

In [ ]:
# Without widgets: no motor exchange, resident robot, robot 1.5x slower, 13 m/s wind
run(motor=False, resident=True, robot_speed=1.5, hs_m=1.5, wind_ms=13.0)

## 4. What this lab cannot tell you

- **Whether the robots can do each step.** The twin has physics for one of them, the coolant-filter change. It succeeds on a redesigned pump skid and fails on the skid as built.
- **What the inspection finds.** Blade-survey and receptor-test durations come from vendor figures and our estimates, not from detection rates.
- **Real weather windows.** A day has changing weather; here it is one sea state and one wind speed for the whole visit.
- **Costs, vessel day rates and spare-part logistics.**

Plan: `CuxhavenSiteTwin/offshore/missions/m1_service_visit.py`. What parts can come out at all: the replacement-matrix lab.